# Pré-processamento: tempo de UTI em insuficiência cardíaca

Pergunta: é possível prever o tempo total de permanência na UTI de pacientes com insuficiência cardíaca usando só o que se sabe nas primeiras 24 horas?

O diagnóstico de alta (CID-10 **I50** ou CID-9 **428**) escolhe a coorte. Ele não entra na matriz do modelo. O alvo `los_dias` é `(outtime - intime)` da UTI.

Rode as células de cima para baixo. Este notebook lê os arquivos reais em `data\`. O CID só filtra a coorte; o alvo é `los_dias`.


## Configuração

Caminhos dos arquivos já baixados em `data\`. O `chartevents` fica em `.csv.gz` (não precisa extrair).


In [ ]:
from pathlib import Path

RANDOM_STATE = 42
FRACAO_TESTE = 0.20

# Pasta com patients, admissions, diagnoses_icd, icustays, labevents e chartevents.csv.gz
MIMIC_ROOT = Path(r"data\mimic-iv")

# Diagnosticos de alta (filtro da coorte I50) — MIMIC-IV-ECG-Ext-ICD
ECG_CSV = Path(r"data\mimic-iv-ecg-ext-icd\1.0.1\records_w_diag_icd10.csv")

# FEVE — MIMIC-IV-Echo
ECHO_MEDIDAS = Path(r"data\mimic-iv-structured-measurement\1.0.1\structured-measurement.csv")
ECHO_ESTUDOS = Path(r"data\mimic-iv-echo-study-list\1.0.1\echo-study-list.csv")

PASTA_SAIDA = Path("artefatos")

for rotulo, caminho in [
    ("MIMIC_ROOT", MIMIC_ROOT),
    ("ECG_CSV", ECG_CSV),
    ("ECHO_MEDIDAS", ECHO_MEDIDAS),
    ("ECHO_ESTUDOS", ECHO_ESTUDOS),
]:
    if not caminho.exists():
        raise FileNotFoundError(f"{rotulo} nao encontrado: {caminho.resolve()}")

if not any(
    (MIMIC_ROOT / nome).exists()
    for nome in ("chartevents.csv.gz", "chartevents.csv")
):
    raise FileNotFoundError(
        f"chartevents nao encontrado em {MIMIC_ROOT.resolve()}. "
        "Salve chartevents.csv.gz nessa pasta (nao precisa extrair)."
    )

print("Caminhos OK:")
print(" ", MIMIC_ROOT.resolve())
print(" ", ECG_CSV.resolve())
print(" ", ECHO_MEDIDAS.resolve())
print(" ", ECHO_ESTUDOS.resolve())


## Bibliotecas e parâmetros clínicos


In [ ]:
from __future__ import annotations

import json
import logging
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from scipy import stats as scipy_stats
except ImportError:
    scipy_stats = None

try:
    from IPython.display import display
except ImportError:
    display = print

HORAS_OBSERVACAO = 24

# Eco conhecido na admissao (ate 48 h antes) ou medido dentro da janela de
# observacao. Um eco feito depois de intime + 24 h nao estaria disponivel
# no momento da predicao e por isso fica de fora.
JANELA_ECO_ANTES = pd.Timedelta(days=2)
JANELA_ECO_DEPOIS = pd.Timedelta(hours=HORAS_OBSERVACAO)

# Familia I50 no CID-10-CM. Internacoes anteriores a transicao do MIMIC
# ainda usam CID-9: 428.x e a mesma doenca, nao um filtro extra.
PREFIXOS_CID10_IC = ("I50",)
PREFIXOS_CID9_IC = ("428",)

# itemid do modulo icu (MetaVision), conferencia em d_items.
ITEMID_FREQ_CARDIACA = 220045
ITEMID_FREQ_RESPIRATORIA = 220210
ITEMID_SPO2 = 220277
ITEMID_PAM_INVASIVA = 220052
ITEMID_PAM_NAO_INVASIVA = 220181
ITEMIDS_VITAIS = {
    ITEMID_FREQ_CARDIACA,
    ITEMID_FREQ_RESPIRATORIA,
    ITEMID_SPO2,
    ITEMID_PAM_INVASIVA,
    ITEMID_PAM_NAO_INVASIVA,
}

# itemid do modulo hosp/labevents (sangue / plasma).
ITEMID_CREATININA = 50912
ITEMID_UREIA = 51006
ITEMID_SODIO = 50983
ITEMID_POTASSIO = 50971
ITEMID_GLICOSE = 50931
ITEMID_PTT = 51275
ITEMIDS_LABS = {
    ITEMID_CREATININA,
    ITEMID_UREIA,
    ITEMID_SODIO,
    ITEMID_POTASSIO,
    ITEMID_GLICOSE,
    ITEMID_PTT,
}

# Nomes observados no MIMIC-IV-Echo. O laboratorio trocou de sistema e o
# mesmo conceito aparece com rotulos diferentes. lvef_upper e um limite,
# nao a fracao de ejecao, e e ignorado.
NOMES_FEVE = ("biplane_lvef", "lvef", "lvef_3d", "ef", "ejection_fraction", "lv_ef")
PRIORIDADE_FEVE = {nome: ordem for ordem, nome in enumerate(NOMES_FEVE)}

# Fora destas faixas o numero e tratado como erro de registro (ausente),
# nao como valor clinico extremo.
FAIXAS_PLAUSIVEIS = {
    "frequencia_cardiaca_24h": (20.0, 280.0),
    "frequencia_respiratoria_24h": (4.0, 70.0),
    "spo2_24h": (50.0, 100.0),
    "pam_24h": (20.0, 250.0),
    "creatinina_24h": (0.1, 30.0),
    "ureia_24h": (1.0, 300.0),
    "sodio_24h": (100.0, 180.0),
    "potassio_24h": (1.5, 9.0),
    "glicose_24h": (20.0, 1500.0),
    "ptt_24h": (10.0, 200.0),
    "feve": (5.0, 85.0),
}

COLUNAS_ID = ["subject_id", "hadm_id", "stay_id"]
COLUNAS_CATEGORICAS = ["genero", "tipo_admissao"]
COLUNAS_NUMERICAS = [
    "idade",
    "pam_24h",
    "frequencia_cardiaca_24h",
    "frequencia_respiratoria_24h",
    "spo2_24h",
    "creatinina_24h",
    "ureia_24h",
    "sodio_24h",
    "potassio_24h",
    "glicose_24h",
    "ptt_24h",
    "feve",
]
COLUNAS_BINARIAS = ["feve_ausente"]
COLUNA_ALVO = "los_dias"

BLOCO_LEITURA = 1_000_000
PADRAO_CID10_IC = re.compile(r"(?:^|[^A-Z0-9])(" + "|".join(PREFIXOS_CID10_IC) + r")")

LOG = logging.getLogger("preprocessamento_los")


# -----------------------------------------------------------------------------
# Utilidades de leitura
# -----------------------------------------------------------------------------

def configurar_log() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%H:%M:%S",
    )

def resolver_arquivo(candidatos: list[Path]) -> Path | None:
    for caminho in candidatos:
        if caminho is not None and caminho.exists():
            return caminho
    return None


def arquivo_mimic(raiz: Path, relativo: str) -> Path:
    """Localiza CSV do MIMIC-IV na arvore hosp/icu ou na pasta plana data/mimic-iv."""
    relativo = relativo.replace("\\", "/")
    nome = Path(relativo).name
    caminho = resolver_arquivo(
        [
            raiz / f"{relativo}.gz",
            raiz / relativo,
            raiz / f"{nome}.gz",
            raiz / nome,
        ]
    )
    if caminho is None:
        raise FileNotFoundError(
            f"Nao encontrei {nome} em {raiz}. "
            f"Coloque o arquivo em {raiz / nome} "
            f"ou na arvore oficial ({raiz / relativo})."
        )
    return caminho


def exigir_colunas(tabela: pd.DataFrame, colunas: list[str], origem: str) -> None:
    faltando = [coluna for coluna in colunas if coluna not in tabela.columns]
    if faltando:
        raise ValueError(
            f"{origem} nao tem as colunas {faltando}. Colunas lidas: {list(tabela.columns)}"
        )


def ler_tabela(caminho: Path, usecols: list[str], parse_dates: list[str] | None = None) -> pd.DataFrame:
    LOG.info("Lendo %s", caminho)
    tabela = pd.read_csv(
        caminho,
        usecols=lambda coluna: coluna in set(usecols),
        parse_dates=parse_dates,
        low_memory=False,
    )
    exigir_colunas(tabela, usecols, str(caminho))
    return tabela


def ler_em_blocos(
    caminho: Path,
    usecols: list[str],
    manter,
    parse_dates: list[str] | None = None,
) -> pd.DataFrame:
    """Varre CSV grande e conserva so as linhas que passam em `manter`."""
    LOG.info("Lendo em blocos %s", caminho)
    partes: list[pd.DataFrame] = []
    linhas_lidas = 0
    leitor = pd.read_csv(
        caminho,
        usecols=lambda coluna: coluna in set(usecols),
        parse_dates=parse_dates,
        chunksize=BLOCO_LEITURA,
        low_memory=False,
    )
    for bloco in leitor:
        linhas_lidas += len(bloco)
        filtrado = manter(bloco)
        if len(filtrado):
            partes.append(filtrado)
        LOG.info("  %s linhas lidas, %s retidas", f"{linhas_lidas:,}", f"{sum(len(p) for p in partes):,}")
    if not partes:
        return pd.DataFrame(columns=usecols)
    return pd.concat(partes, ignore_index=True)


def aplicar_faixa(serie: pd.Series, minimo: float, maximo: float) -> pd.Series:
    valores = pd.to_numeric(serie, errors="coerce")
    return valores.where((valores >= minimo) & (valores <= maximo))


def codificar_one_hot():
    """OneHotEncoder compativel com sklearn 1.2+ (sparse_output) e versoes anteriores."""
    parametros = dict(handle_unknown="ignore", dtype=np.float64)
    try:
        return OneHotEncoder(sparse_output=False, **parametros)
    except TypeError:
        return OneHotEncoder(sparse=False, **parametros)


def p_valor_pearson(r: float, n: int) -> float:
    if n < 3 or not np.isfinite(r) or abs(r) >= 1.0:
        return float("nan")
    t = r * np.sqrt((n - 2) / (1.0 - r * r))
    if scipy_stats is not None:
        return float(2.0 * scipy_stats.t.sf(abs(t), df=n - 2))
    # Aproximacao normal do teste t, adequada para n grande desta coorte.
    from math import erfc, sqrt

    return float(erfc(abs(t) / sqrt(2.0)))

def salvar_csv(tabela: pd.DataFrame, caminho: Path) -> None:
    tabela.to_csv(caminho, index=False, encoding="utf-8-sig")
    LOG.info("Gravado %s", caminho)


## 1. Carregamento, cruzamento e limpeza

`subject_id` liga o paciente. `hadm_id` liga a estadia de UTI ao diagnóstico de alta. A fração de ejeção vem do ecocardiograma estruturado: o exame mais próximo do início da UTI, entre 48 horas antes e 24 horas depois. Sinais vitais e exames laboratoriais são a mediana dentro das primeiras 24 horas.


In [ ]:
def hadm_ids_insuficiencia_cardiaca(caminho_diagnosticos: Path) -> set[int]:
    """Internacoes com alta hospitalar por IC (I50 no CID-10 ou 428 no CID-9)."""
    diagnosticos = ler_tabela(
        caminho_diagnosticos,
        ["hadm_id", "icd_code", "icd_version"],
    )
    codigo = (
        diagnosticos["icd_code"]
        .astype("string")
        .str.upper()
        .str.replace(".", "", regex=False)
        .str.strip()
    )
    versao = pd.to_numeric(diagnosticos["icd_version"], errors="coerce")
    mascara_cid10 = (versao == 10) & codigo.str.startswith(PREFIXOS_CID10_IC)
    mascara_cid9 = (versao == 9) & codigo.str.startswith(PREFIXOS_CID9_IC)
    hadm = diagnosticos.loc[mascara_cid10 | mascara_cid9, "hadm_id"].dropna().astype(np.int64)
    encontrados = set(hadm.tolist())
    LOG.info("Internacoes com IC em diagnoses_icd: %s", f"{len(encontrados):,}")
    return encontrados


def hadm_ids_ic_no_ecg(caminho_ecg: Path | None) -> set[int]:
    """
    Segunda fonte do mesmo criterio, vinda de records_w_diag_icd10.csv.

    all_diag_hosp junta os CID-10 de alta hospitalar ja padronizados pelo
    dataset de ECG. O laudo do ECG nao vira feature: esta tabela so confirma
    o hadm_id da coorte.
    """
    if caminho_ecg is None:
        LOG.info("Arquivo de ECG nao informado. A coorte usa apenas diagnoses_icd.")
        return set()

    encontrados: set[int] = set()
    leitor = pd.read_csv(
        caminho_ecg,
        usecols=lambda coluna: coluna in {"hosp_hadm_id", "ed_hadm_id", "all_diag_hosp"},
        chunksize=200_000,
        low_memory=False,
    )
    for bloco in leitor:
        exigir_colunas(bloco, ["all_diag_hosp"], str(caminho_ecg))
        hadm = pd.Series(np.nan, index=bloco.index, dtype="float64")
        if "hosp_hadm_id" in bloco.columns:
            hadm = pd.to_numeric(bloco["hosp_hadm_id"], errors="coerce")
        if "ed_hadm_id" in bloco.columns:
            hadm = hadm.fillna(pd.to_numeric(bloco["ed_hadm_id"], errors="coerce"))
        texto = (
            bloco["all_diag_hosp"]
            .astype("string")
            .str.upper()
            .str.replace(".", "", regex=False)
            .fillna("")
        )
        mascara = texto.str.contains(PADRAO_CID10_IC, na=False)
        encontrados.update(hadm.loc[mascara].dropna().astype(np.int64).tolist())
    LOG.info("Internacoes com I50 em all_diag_hosp (ECG-Ext-ICD): %s", f"{len(encontrados):,}")
    return encontrados


def montar_estadias_ic(
    raiz_mimic: Path,
    hadm_ic: set[int],
) -> pd.DataFrame:
    """Uma linha por estadia de UTI, restrita a internacoes com IC."""
    estadias = ler_tabela(
        arquivo_mimic(raiz_mimic, "icu/icustays.csv"),
        ["subject_id", "hadm_id", "stay_id", "intime", "outtime"],
        parse_dates=["intime", "outtime"],
    )
    admissoes = ler_tabela(
        arquivo_mimic(raiz_mimic, "hosp/admissions.csv"),
        ["hadm_id", "admittime", "admission_type"],
        parse_dates=["admittime"],
    )
    pacientes = ler_tabela(
        arquivo_mimic(raiz_mimic, "hosp/patients.csv"),
        ["subject_id", "gender", "anchor_age", "anchor_year"],
    )

    estadias = estadias[estadias["hadm_id"].isin(hadm_ic)].copy()
    if estadias.empty:
        raise RuntimeError(
            "Nenhuma estadia de UTI restou depois do filtro de insuficiencia cardiaca. "
            "Confira a versao do MIMIC-IV e os arquivos de diagnostico."
        )

    base = estadias.merge(
        admissoes[["hadm_id", "admittime", "admission_type"]],
        on="hadm_id",
        how="left",
        validate="many_to_one",
    )
    base = base.merge(
        pacientes[["subject_id", "gender", "anchor_age", "anchor_year"]],
        on="subject_id",
        how="left",
        validate="many_to_one",
    )

    # O deslocamento de datas do MIMIC e o mesmo dentro do paciente, entao a
    # diferenca de anos entre admittime e anchor_year recupera a idade real.
    # Idade acima de 89 anos e agrupada em 91 no processo de deidentificacao.
    base["idade"] = (
        pd.to_numeric(base["anchor_age"], errors="coerce")
        + base["admittime"].dt.year
        - pd.to_numeric(base["anchor_year"], errors="coerce")
    )
    base["idade"] = base["idade"].clip(upper=91)

    # LOS da UTI em dias fracionarios. icustays.los ja e essa conta; ela e
    # refeita aqui para o alvo ficar explicito no pipeline.
    base["los_dias"] = (base["outtime"] - base["intime"]) / pd.Timedelta(days=1)

    base["genero"] = base["gender"].astype("string").str.upper().str.strip()
    base["tipo_admissao"] = base["admission_type"].astype("string").str.strip()
    base.loc[~base["genero"].isin(["M", "F"]), "genero"] = pd.NA

    antes = len(base)
    base = base[
        base["idade"].between(18, 91)
        & base["los_dias"].gt(0)
        & base["intime"].notna()
        & base["outtime"].notna()
    ].copy()
    LOG.info(
        "Estadias de UTI com IC: %s (removidas %s sem idade adulta ou LOS valido)",
        f"{len(base):,}",
        f"{antes - len(base):,}",
    )
    return base


def agregar_sinais_vitais(caminho_chartevents: Path, estadias: pd.DataFrame) -> pd.DataFrame:
    """Mediana de cada sinal vital entre intime e intime + 24 h."""
    ids_estadia = set(estadias["stay_id"].astype(np.int64).tolist())
    inicio = estadias.set_index("stay_id")["intime"]

    def manter(bloco: pd.DataFrame) -> pd.DataFrame:
        item = pd.to_numeric(bloco["itemid"], errors="coerce")
        estadia = pd.to_numeric(bloco["stay_id"], errors="coerce")
        valor = pd.to_numeric(bloco["valuenum"], errors="coerce")
        mascara = item.isin(ITEMIDS_VITAIS) & estadia.isin(ids_estadia) & valor.notna()
        return bloco.loc[mascara, ["stay_id", "charttime", "itemid", "valuenum"]]

    bruto = ler_em_blocos(
        caminho_chartevents,
        ["stay_id", "charttime", "itemid", "valuenum"],
        manter,
        parse_dates=["charttime"],
    )
    if bruto.empty:
        LOG.warning("Nenhum sinal vital retido. As colunas vitais ficarao ausentes.")
        return pd.DataFrame({"stay_id": estadias["stay_id"]})

    bruto["stay_id"] = bruto["stay_id"].astype(np.int64)
    bruto["itemid"] = bruto["itemid"].astype(np.int64)
    bruto["intime"] = bruto["stay_id"].map(inicio)
    fim = bruto["intime"] + pd.Timedelta(hours=HORAS_OBSERVACAO)
    bruto = bruto[(bruto["charttime"] >= bruto["intime"]) & (bruto["charttime"] < fim)]

    def mediana(itemid: int) -> pd.Series:
        return (
            bruto.loc[bruto["itemid"] == itemid]
            .groupby("stay_id")["valuenum"]
            .median()
        )

    pam_invasiva = mediana(ITEMID_PAM_INVASIVA)
    pam_nao_invasiva = mediana(ITEMID_PAM_NAO_INVASIVA)
    vitais = pd.DataFrame(index=estadias["stay_id"].astype(np.int64).unique())
    vitais.index.name = "stay_id"
    vitais["pam_24h"] = pam_invasiva.combine_first(pam_nao_invasiva)
    vitais["frequencia_cardiaca_24h"] = mediana(ITEMID_FREQ_CARDIACA)
    vitais["frequencia_respiratoria_24h"] = mediana(ITEMID_FREQ_RESPIRATORIA)
    vitais["spo2_24h"] = mediana(ITEMID_SPO2)
    return vitais.reset_index()


def agregar_laboratorios(caminho_labevents: Path, estadias: pd.DataFrame) -> pd.DataFrame:
    """Mediana de cada exame entre intime e intime + 24 h."""
    ids_paciente = set(estadias["subject_id"].astype(np.int64).tolist())
    janelas = estadias[["subject_id", "stay_id", "intime"]].copy()
    janelas["subject_id"] = janelas["subject_id"].astype(np.int64)
    janelas["fim"] = janelas["intime"] + pd.Timedelta(hours=HORAS_OBSERVACAO)

    def manter(bloco: pd.DataFrame) -> pd.DataFrame:
        item = pd.to_numeric(bloco["itemid"], errors="coerce")
        paciente = pd.to_numeric(bloco["subject_id"], errors="coerce")
        valor = pd.to_numeric(bloco["valuenum"], errors="coerce")
        mascara = item.isin(ITEMIDS_LABS) & paciente.isin(ids_paciente) & valor.notna()
        return bloco.loc[mascara, ["subject_id", "charttime", "itemid", "valuenum"]]

    bruto = ler_em_blocos(
        caminho_labevents,
        ["subject_id", "charttime", "itemid", "valuenum"],
        manter,
        parse_dates=["charttime"],
    )
    colunas_lab = [
        "stay_id",
        "creatinina_24h",
        "ureia_24h",
        "sodio_24h",
        "potassio_24h",
        "glicose_24h",
        "ptt_24h",
    ]
    if bruto.empty:
        LOG.warning("Nenhum laboratorio retido. As colunas de exame ficarao ausentes.")
        return pd.DataFrame({"stay_id": estadias["stay_id"]})

    bruto["subject_id"] = bruto["subject_id"].astype(np.int64)
    bruto["itemid"] = bruto["itemid"].astype(np.int64)
    # Cada exame pode cruzar com mais de uma estadia do mesmo paciente.
    # A janela [intime, intime + 24 h) separa as estadias, que nao se sobrepoem.
    cruzado = bruto.merge(janelas, on="subject_id", how="inner")
    cruzado = cruzado[
        (cruzado["charttime"] >= cruzado["intime"]) & (cruzado["charttime"] < cruzado["fim"])
    ]

    nome_por_item = {
        ITEMID_CREATININA: "creatinina_24h",
        ITEMID_UREIA: "ureia_24h",
        ITEMID_SODIO: "sodio_24h",
        ITEMID_POTASSIO: "potassio_24h",
        ITEMID_GLICOSE: "glicose_24h",
        ITEMID_PTT: "ptt_24h",
    }
    if cruzado.empty:
        tabela = pd.DataFrame({"stay_id": estadias["stay_id"].astype(np.int64)})
        for coluna in colunas_lab:
            if coluna != "stay_id":
                tabela[coluna] = np.nan
        return tabela[colunas_lab]

    cruzado["exame"] = cruzado["itemid"].map(nome_por_item)
    tabela = (
        cruzado.pivot_table(
            index="stay_id",
            columns="exame",
            values="valuenum",
            aggfunc="median",
        )
        .reset_index()
    )
    for coluna in colunas_lab:
        if coluna not in tabela.columns:
            tabela[coluna] = np.nan
    return tabela[colunas_lab]


def _normalizar_feve(valores: pd.Series) -> pd.Series:
    """Converte fracao (0,55) em porcentagem (55) e descarta fora da faixa."""
    numero = pd.to_numeric(valores, errors="coerce")
    numero = numero.where(numero > 1.5, numero * 100.0)
    minimo, maximo = FAIXAS_PLAUSIVEIS["feve"]
    return numero.where((numero >= minimo) & (numero <= maximo))


def anexar_feve(
    caminho_medidas: Path | None,
    caminho_estudos: Path | None,
    estadias: pd.DataFrame,
) -> pd.DataFrame:
    """
    FEVE do estudo de eco mais proximo do intime, dentro da janela permitida.

    O valor vem de structured_measurement (formato longo). echo-study-list
    acrescenta o study_id do DICOM quando o mesmo measurement_id cai na janela
    de 2 dias documentada pelo modulo Echo. Esse study_id e rastreio, nao feature.
    """
    vazio = estadias[["stay_id"]].copy()
    vazio["feve"] = np.nan
    vazio["echo_measurement_id"] = pd.NA
    vazio["echo_study_id"] = pd.NA
    if caminho_medidas is None:
        LOG.info("Medidas de eco nao informadas. FEVE ficara ausente.")
        return vazio

    candidatos_nome = set(NOMES_FEVE)

    def manter(bloco: pd.DataFrame) -> pd.DataFrame:
        nome = bloco["measurement"].astype("string").str.lower().str.strip()
        return bloco.loc[nome.isin(candidatos_nome)]

    medidas = ler_em_blocos(
        caminho_medidas,
        ["subject_id", "measurement_id", "measurement_datetime", "test_type", "measurement", "result"],
        manter,
        parse_dates=["measurement_datetime"],
    )
    if medidas.empty:
        LOG.warning("Nenhuma FEVE encontrada em %s", caminho_medidas.name)
        return vazio

    medidas["measurement"] = medidas["measurement"].astype("string").str.lower().str.strip()
    medidas["feve"] = _normalizar_feve(medidas["result"])
    medidas = medidas.dropna(subset=["feve", "measurement_datetime", "subject_id"])
    medidas["subject_id"] = medidas["subject_id"].astype(np.int64)
    medidas["prioridade"] = medidas["measurement"].map(PRIORIDADE_FEVE)
    medidas["preferir_tte"] = (
        medidas["test_type"].astype("string").str.upper().eq("TTE").map({True: 0, False: 1})
    )
    # Um estudo (measurement_id) pode registrar mais de um nome de FEVE.
    medidas = (
        medidas.sort_values(["subject_id", "measurement_id", "prioridade", "preferir_tte"])
        .drop_duplicates(["subject_id", "measurement_id"], keep="first")
    )

    pacientes_coorte = set(estadias["subject_id"].astype(np.int64).tolist())
    medidas = medidas[medidas["subject_id"].isin(pacientes_coorte)]
    if medidas.empty:
        LOG.info("Nenhum exame de eco pertence a coorte de IC.")
        return vazio

    # A janela e assimetrica: eco de ate 48 h antes da UTI pode ser conhecido
    # na admissao; eco posterior so entra se caiu nas primeiras 24 h. Por isso
    # a escolha e feita depois do filtro, e nao pelo vizinho temporal mais
    # proximo numa janela simetrica.
    esquerda = estadias[["stay_id", "subject_id", "intime"]].copy()
    esquerda["subject_id"] = esquerda["subject_id"].astype(np.int64)
    cruzado = esquerda.merge(
        medidas[["subject_id", "measurement_datetime", "measurement_id", "feve"]],
        on="subject_id",
        how="inner",
    )
    delta = cruzado["measurement_datetime"] - cruzado["intime"]
    cruzado = cruzado.loc[delta.between(-JANELA_ECO_ANTES, JANELA_ECO_DEPOIS)].copy()
    cruzado["distancia"] = delta.loc[cruzado.index].abs()
    cruzado = (
        cruzado.sort_values(["stay_id", "distancia"])
        .drop_duplicates("stay_id", keep="first")
    )
    ligado = esquerda.merge(
        cruzado[["stay_id", "measurement_datetime", "measurement_id", "feve"]],
        on="stay_id",
        how="left",
    )

    if caminho_estudos is not None:
        estudos = pd.read_csv(caminho_estudos, low_memory=False)
        if "measurement_id" in estudos.columns and "study_id" in estudos.columns:
            mapa = (
                estudos[["measurement_id", "study_id"]]
                .dropna(subset=["measurement_id"])
                .drop_duplicates("measurement_id")
            )
            ligado = ligado.merge(mapa, on="measurement_id", how="left")
            ligado = ligado.rename(columns={"study_id": "echo_study_id"})
        else:
            LOG.warning(
                "echo-study-list sem measurement_id/study_id. Colunas: %s",
                list(estudos.columns),
            )
            ligado["echo_study_id"] = pd.NA
    else:
        ligado["echo_study_id"] = pd.NA

    LOG.info(
        "Estadias com FEVE na janela: %s de %s",
        f"{ligado['feve'].notna().sum():,}",
        f"{len(ligado):,}",
    )
    return ligado[["stay_id", "feve", "measurement_id", "echo_study_id"]].rename(
        columns={"measurement_id": "echo_measurement_id"}
    )


def marcar_valores_implausiveis(coorte: pd.DataFrame) -> pd.DataFrame:
    limpa = coorte.copy()
    for coluna, (minimo, maximo) in FAIXAS_PLAUSIVEIS.items():
        if coluna == "feve" or coluna not in limpa.columns:
            continue
        antes = limpa[coluna].notna().sum()
        limpa[coluna] = aplicar_faixa(limpa[coluna], minimo, maximo)
        removidos = antes - limpa[coluna].notna().sum()
        if removidos:
            LOG.info("%s: %s valores fora de [%s, %s] viraram ausentes", coluna, removidos, minimo, maximo)
    limpa["feve_ausente"] = limpa["feve"].isna().astype(np.int8)
    return limpa


def construir_coorte_mimic(
    raiz_mimic: Path,
    caminho_ecg: Path | None,
    caminho_medidas: Path | None,
    caminho_estudos: Path | None,
) -> pd.DataFrame:
    caminho_dx = arquivo_mimic(raiz_mimic, "hosp/diagnoses_icd.csv")

    hadm_ic = hadm_ids_insuficiencia_cardiaca(caminho_dx)
    hadm_ecg = hadm_ids_ic_no_ecg(caminho_ecg)
    so_ecg = hadm_ecg - hadm_ic
    hadm_ic = hadm_ic | hadm_ecg
    LOG.info(
        "Uniao das fontes de diagnostico: %s internacoes (%s apenas no ECG-Ext-ICD)",
        f"{len(hadm_ic):,}",
        f"{len(so_ecg):,}",
    )

    estadias = montar_estadias_ic(raiz_mimic, hadm_ic)
    fonte = np.where(
        estadias["hadm_id"].isin(hadm_ecg) & estadias["hadm_id"].isin(hadm_ic - so_ecg),
        "diagnoses_icd+ecg",
        np.where(estadias["hadm_id"].isin(so_ecg), "ecg", "diagnoses_icd"),
    )
    estadias["fonte_diagnostico_ic"] = fonte

    vitais = agregar_sinais_vitais(arquivo_mimic(raiz_mimic, "icu/chartevents.csv"), estadias)
    labs = agregar_laboratorios(arquivo_mimic(raiz_mimic, "hosp/labevents.csv"), estadias)
    feve = anexar_feve(caminho_medidas, caminho_estudos, estadias)

    coorte = estadias.merge(vitais, on="stay_id", how="left", validate="one_to_one")
    coorte = coorte.merge(labs, on="stay_id", how="left", validate="one_to_one")
    coorte = coorte.merge(feve, on="stay_id", how="left", validate="one_to_one")
    coorte = marcar_valores_implausiveis(coorte)

    colunas = (
        COLUNAS_ID
        + ["fonte_diagnostico_ic", "echo_measurement_id", "echo_study_id"]
        + COLUNAS_CATEGORICAS
        + COLUNAS_NUMERICAS
        + COLUNAS_BINARIAS
        + [COLUNA_ALVO]
    )
    for coluna in colunas:
        if coluna not in coorte.columns:
            coorte[coluna] = pd.NA
    return coorte[colunas].sort_values(["subject_id", "stay_id"]).reset_index(drop=True)


A função abaixo só é usada no modo demonstração. Quando `MODO_DEMO` é `False`, a coorte vem de `construir_coorte_mimic`.


In [ ]:
def coorte_sintetica(n_estadias: int = 4000, random_state: int = RANDOM_STATE) -> pd.DataFrame:
    """
    Coorte com o mesmo esquema da base real, para exercitar o pre-processamento
    antes de o download do MIMIC terminar. Nao usar estes arquivos no TCC.
    """
    rng = np.random.default_rng(random_state)
    n_pacientes = int(n_estadias * 0.82)
    subject_id = rng.integers(10_000, 10_000 + n_pacientes, size=n_estadias)
    stay_id = np.arange(300_000, 300_000 + n_estadias)
    hadm_id = stay_id + 7

    idade = np.clip(rng.normal(72, 12, size=n_estadias), 18, 91)
    genero = rng.choice(["M", "F"], size=n_estadias, p=[0.54, 0.46])
    tipo_admissao = rng.choice(
        ["EW EMER.", "URGENT", "ELECTIVE", "OBSERVATION ADMIT"],
        size=n_estadias,
        p=[0.62, 0.22, 0.08, 0.08],
    )
    pam = np.clip(rng.normal(78, 14, size=n_estadias), 40, 140)
    fc = np.clip(rng.normal(92, 18, size=n_estadias), 40, 160)
    fr = np.clip(rng.normal(20, 5, size=n_estadias), 8, 40)
    spo2 = np.clip(rng.normal(95, 3, size=n_estadias), 70, 100)
    creatinina = np.clip(rng.lognormal(np.log(1.3), 0.45, size=n_estadias), 0.3, 12)
    ureia = np.clip(18 + 14 * creatinina + rng.normal(0, 8, size=n_estadias), 4, 180)
    sodio = np.clip(rng.normal(137, 4, size=n_estadias), 115, 160)
    potassio = np.clip(rng.normal(4.2, 0.6, size=n_estadias), 2.2, 7.5)
    glicose = np.clip(rng.normal(145, 50, size=n_estadias), 50, 500)
    ptt = np.clip(rng.normal(34, 10, size=n_estadias), 18, 120)
    feve = np.clip(rng.normal(42, 14, size=n_estadias), 10, 75)

    emergencia = (tipo_admissao == "EW EMER.").astype(float)
    sinal = (
        1.6
        + 0.035 * (idade - 70)
        + 0.85 * (creatinina - 1.2)
        + 0.02 * (ureia - 25)
        + 0.012 * (fc - 90)
        + 0.04 * (fr - 18)
        - 0.018 * (pam - 80)
        - 0.025 * (feve - 45)
        - 0.06 * (spo2 - 95)
        + 0.35 * emergencia
        + rng.normal(0, 1.1, size=n_estadias)
    )
    los_dias = np.clip(np.exp(sinal / 3.2), 0.2, 40)

    coorte = pd.DataFrame(
        {
            "subject_id": subject_id,
            "hadm_id": hadm_id,
            "stay_id": stay_id,
            "fonte_diagnostico_ic": "sintetica",
            "echo_measurement_id": pd.NA,
            "echo_study_id": pd.NA,
            "genero": genero,
            "tipo_admissao": tipo_admissao,
            "idade": idade,
            "pam_24h": pam,
            "frequencia_cardiaca_24h": fc,
            "frequencia_respiratoria_24h": fr,
            "spo2_24h": spo2,
            "creatinina_24h": creatinina,
            "ureia_24h": ureia,
            "sodio_24h": sodio,
            "potassio_24h": potassio,
            "glicose_24h": glicose,
            "ptt_24h": ptt,
            "feve": feve,
            "los_dias": los_dias,
        }
    )

    # Ausencia no mesmo espirito da base real: FEVE falta com frequencia,
    # exames laboratoriais faltam pouco, alguns categoricos ficam em branco.
    for coluna, taxa in {
        "feve": 0.35,
        "ptt_24h": 0.18,
        "ureia_24h": 0.08,
        "creatinina_24h": 0.05,
        "pam_24h": 0.06,
        "glicose_24h": 0.07,
        "tipo_admissao": 0.02,
    }.items():
        apagar = rng.random(n_estadias) < taxa
        coorte.loc[apagar, coluna] = np.nan

    coorte["feve_ausente"] = coorte["feve"].isna().astype(np.int8)
    coorte["tipo_admissao"] = coorte["tipo_admissao"].astype("string")
    LOG.info("Coorte sintetica: %s estadias, %s pacientes", f"{len(coorte):,}", f"{coorte['subject_id'].nunique():,}")
    return coorte


def resumo_ausentes(coorte: pd.DataFrame) -> pd.DataFrame:
    colunas = COLUNAS_CATEGORICAS + COLUNAS_NUMERICAS + COLUNAS_BINARIAS
    resumo = pd.DataFrame(
        {
            "variavel": colunas,
            "n_ausentes": [int(coorte[coluna].isna().sum()) for coluna in colunas],
            "pct_ausentes": [float(coorte[coluna].isna().mean()) for coluna in colunas],
        }
    )
    return resumo.sort_values("pct_ausentes", ascending=False)


## Execução da etapa 1

Leitura, filtro da coorte, painel das primeiras 24 horas e contagem de valores ausentes. A imputação ainda não acontece nesta célula.


In [ ]:
saida = Path(PASTA_SAIDA)
saida.mkdir(parents=True, exist_ok=True)
configurar_log()

# Etapa 1. Filtro da coorte pelo CID (I50/428), join, painel 24h e alvo los_dias.
# A mediana dos ausentes so e calculada depois da divisao.
coorte = construir_coorte_mimic(
    raiz_mimic=Path(MIMIC_ROOT),
    caminho_ecg=Path(ECG_CSV),
    caminho_medidas=Path(ECHO_MEDIDAS),
    caminho_estudos=Path(ECHO_ESTUDOS),
)

ausentes = resumo_ausentes(coorte)
salvar_csv(coorte, saida / "coorte_bruta.csv")
salvar_csv(ausentes, saida / "ausentes_antes_da_imputacao.csv")
print(
    f"Estadias: {len(coorte):,} | pacientes: {coorte['subject_id'].nunique():,} | "
    f"LOS mediano: {coorte[COLUNA_ALVO].median():.2f} dias"
)
display(coorte.head())
display(ausentes)


## 2 a 4. Divisão, imputação, one-hot, Pearson e escala

A separação 80/20 é por paciente: as estadias da mesma pessoa não se misturam entre treino e teste. Mediana, one-hot e escala são aprendidos no treino e só aplicados no teste.

O one-hot transforma gênero e tipo de admissão em colunas 0/1, sem inventar ordem entre as categorias.

O teste de Pearson mede a associação linear de cada variável com `los_dias` no treino. Ele não remove variável, porque XGBoost e Random Forest usam relações não lineares.

`X_treino` fica sem escala, para as árvores. `X_treino_escalonado` padroniza só as variáveis contínuas, para Ridge e Lasso.


In [ ]:
def dividir_por_paciente(
    coorte: pd.DataFrame,
    fracao_teste: float,
    random_state: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    80/20 no nivel do paciente. Todas as estadias de UTI de uma mesma pessoa
    ficam do mesmo lado, para o teste nao ver um sujeito que ja entrou no treino.
    """
    divisor = GroupShuffleSplit(n_splits=1, test_size=fracao_teste, random_state=random_state)
    indice_treino, indice_teste = next(divisor.split(coorte, groups=coorte["subject_id"]))
    treino = coorte.iloc[indice_treino].copy()
    teste = coorte.iloc[indice_teste].copy()
    sobreposicao = set(treino["subject_id"]).intersection(teste["subject_id"])
    if sobreposicao:
        raise RuntimeError("Vazamento de paciente entre treino e teste.")
    LOG.info(
        "Divisao por paciente: treino %s estadias / %s pacientes | teste %s estadias / %s pacientes",
        f"{len(treino):,}",
        f"{treino['subject_id'].nunique():,}",
        f"{len(teste):,}",
        f"{teste['subject_id'].nunique():,}",
    )
    return treino, teste


def imputar_medianas(
    treino: pd.DataFrame,
    teste: pd.DataFrame,
    colunas: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, SimpleImputer]:
    """
    Mediana aprendida so no treino. MICE (IterativeImputer) existe no sklearn,
    mas em dezenas de milhares de estadias a mediana e mais estavel, mais rapida
    e mais facil de descrever no metodo. Coluna inteiramente vazia no treino
    recebe 0 para o imputer nao quebrar.
    """
    treino = treino.copy()
    teste = teste.copy()
    for coluna in colunas:
        if treino[coluna].notna().sum() == 0:
            LOG.warning("Coluna %s sem nenhum valor no treino. Imputacao fixada em 0.", coluna)
            treino[coluna] = 0.0
            teste[coluna] = teste[coluna].fillna(0.0)

    imputer = SimpleImputer(strategy="median")
    treino[colunas] = imputer.fit_transform(treino[colunas])
    teste[colunas] = imputer.transform(teste[colunas])
    return treino, teste, imputer


def aplicar_one_hot(
    treino: pd.DataFrame,
    teste: pd.DataFrame,
    colunas: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, OneHotEncoder]:
    """
    Categoricas viram colunas 0/1. Nao se usa rotulo inteiro (0, 1, 2), porque
    isso inventaria uma ordem entre os tipos de admissao.
    """
    treino = treino.copy()
    teste = teste.copy()
    for coluna in colunas:
        treino[coluna] = treino[coluna].astype("string").fillna("desconhecido")
        teste[coluna] = teste[coluna].astype("string").fillna("desconhecido")

    encoder = codificar_one_hot()
    matriz_treino = encoder.fit_transform(treino[colunas])
    matriz_teste = encoder.transform(teste[colunas])
    nomes = list(encoder.get_feature_names_out(colunas))
    dummies_treino = pd.DataFrame(matriz_treino, columns=nomes, index=treino.index)
    dummies_teste = pd.DataFrame(matriz_teste, columns=nomes, index=teste.index)

    treino = pd.concat([treino.drop(columns=colunas), dummies_treino], axis=1)
    teste = pd.concat([teste.drop(columns=colunas), dummies_teste], axis=1)
    return treino, teste, encoder


def correlacao_pearson(variaveis: pd.DataFrame, alvo: pd.Series) -> pd.DataFrame:
    """
    Pearson entre cada coluna de X e o LOS, calculado apenas no treino.

    Associacao linear fraca nao elimina a variavel: XGBoost e Random Forest
    capturam relacoes nao lineares. A tabela serve de leitura inicial e de
    material para o metodo, nao de filtro automatico.
    """
    linhas = []
    alvo_array = pd.to_numeric(alvo, errors="coerce").to_numpy(dtype=float)
    for coluna in variaveis.columns:
        x = pd.to_numeric(variaveis[coluna], errors="coerce").to_numpy(dtype=float)
        mascara = np.isfinite(x) & np.isfinite(alvo_array)
        n = int(mascara.sum())
        if n < 3 or np.nanstd(x[mascara]) == 0 or np.nanstd(alvo_array[mascara]) == 0:
            r = float("nan")
        else:
            r = float(np.corrcoef(x[mascara], alvo_array[mascara])[0, 1])
        linhas.append(
            {
                "variavel": coluna,
                "n": n,
                "pearson_r": r,
                "abs_r": abs(r) if np.isfinite(r) else np.nan,
                "p_valor": p_valor_pearson(r, n),
            }
        )
    return pd.DataFrame(linhas).sort_values("abs_r", ascending=False, na_position="last")


def escalonar_numericas(
    treino: pd.DataFrame,
    teste: pd.DataFrame,
    colunas: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, StandardScaler]:
    """
    StandardScaler so nas continuas. As dummies do one-hot permanecem 0/1.
    Arvores (XGBoost, Random Forest) usam a matriz sem esta escala.
    Ridge e Lasso usam a matriz escalonada.
    """
    presentes = [coluna for coluna in colunas if coluna in treino.columns]
    scaler = StandardScaler()
    treino_esc = treino.copy()
    teste_esc = teste.copy()
    treino_esc[presentes] = scaler.fit_transform(treino[presentes])
    teste_esc[presentes] = scaler.transform(teste[presentes])
    return treino_esc, teste_esc, scaler


def matriz_de_modelo(tabela: pd.DataFrame, colunas_x: list[str]) -> tuple[pd.DataFrame, pd.Series]:
    x = tabela[colunas_x].apply(pd.to_numeric, errors="coerce")
    y = pd.to_numeric(tabela[COLUNA_ALVO], errors="coerce")
    if x.isna().any().any() or y.isna().any():
        raise RuntimeError("Ainda ha ausentes em X ou y depois da imputacao.")
    if COLUNA_ALVO in x.columns:
        raise RuntimeError("O alvo los_dias entrou na matriz de features.")
    return x, y


### Divisão treino e teste


In [ ]:
# A divisao acontece antes do imputer, do one-hot e do scaler.
treino, teste = dividir_por_paciente(coorte, FRACAO_TESTE, RANDOM_STATE)
salvar_csv(treino[COLUNAS_ID], saida / "ids_treino.csv")
salvar_csv(teste[COLUNAS_ID], saida / "ids_teste.csv")
print(f"Treino: {len(treino):,} estadias | Teste: {len(teste):,} estadias")


### Imputação pela mediana do treino


In [ ]:
treino, teste, imputer = imputar_medianas(treino, teste, COLUNAS_NUMERICAS)
pd.Series(imputer.statistics_, index=COLUNAS_NUMERICAS, name="mediana_treino")


### One-hot encoding


In [ ]:
treino, teste, encoder = aplicar_one_hot(treino, teste, COLUNAS_CATEGORICAS)
colunas_modelo = [
    coluna
    for coluna in treino.columns
    if coluna not in COLUNAS_ID
    + ["fonte_diagnostico_ic", "echo_measurement_id", "echo_study_id", COLUNA_ALVO]
]
colunas_para_escala = [coluna for coluna in COLUNAS_NUMERICAS if coluna in colunas_modelo]

x_treino, y_treino = matriz_de_modelo(treino, colunas_modelo)
x_teste, y_teste = matriz_de_modelo(teste, colunas_modelo)
x_teste = x_teste.reindex(columns=x_treino.columns, fill_value=0.0)
print("Colunas do modelo:")
print(list(x_treino.columns))
display(x_treino.head())


### Correlação de Pearson com o LOS


In [ ]:
pearson = correlacao_pearson(x_treino, y_treino)
salvar_csv(pearson, saida / "correlacao_pearson_treino.csv")
display(pearson)


### Normalização e gravação das matrizes


In [ ]:
x_treino_esc, x_teste_esc, scaler = escalonar_numericas(x_treino, x_teste, colunas_para_escala)

salvar_csv(x_treino, saida / "X_treino.csv")
salvar_csv(x_teste, saida / "X_teste.csv")
salvar_csv(y_treino.to_frame(COLUNA_ALVO), saida / "y_treino.csv")
salvar_csv(y_teste.to_frame(COLUNA_ALVO), saida / "y_teste.csv")
salvar_csv(x_treino_esc, saida / "X_treino_escalonado.csv")
salvar_csv(x_teste_esc, saida / "X_teste_escalonado.csv")

medianas = {
    coluna: float(valor)
    for coluna, valor in zip(COLUNAS_NUMERICAS, imputer.statistics_)
}
metadados = {
    "modo": "mimic",
    "random_state": RANDOM_STATE,
    "fracao_teste": FRACAO_TESTE,
    "n_estadias": int(len(coorte)),
    "n_pacientes": int(coorte["subject_id"].nunique()),
    "n_treino": int(len(x_treino)),
    "n_teste": int(len(x_teste)),
    "horas_observacao": HORAS_OBSERVACAO,
    "janela_eco_antes_horas": JANELA_ECO_ANTES / pd.Timedelta(hours=1),
    "janela_eco_depois_horas": JANELA_ECO_DEPOIS / pd.Timedelta(hours=1),
    "colunas_modelo": colunas_modelo,
    "colunas_escalonadas": colunas_para_escala,
    "mediana_treino": medianas,
    "media_scaler": scaler.mean_.tolist(),
    "desvio_scaler": scaler.scale_.tolist(),
    "categorias_one_hot": {
        coluna: [str(categoria) for categoria in categorias]
        for coluna, categorias in zip(COLUNAS_CATEGORICAS, encoder.categories_)
    },
    "alvo": COLUNA_ALVO,
    "nota_alvo": "Dias fracionarios de UTI, (outtime - intime). Nao e LOS hospitalar.",
    "nota_modelos": (
        "XGBoost e Random Forest consomem X_treino/X_teste. "
        "Ridge e Lasso consomem X_treino_escalonado/X_teste_escalonado."
    ),
}
caminho_meta = saida / "metadados_preprocessamento.json"
caminho_meta.write_text(json.dumps(metadados, indent=2, ensure_ascii=False), encoding="utf-8")

if MODO_DEMO:
    linha_creatinina = pearson.loc[pearson["variavel"] == "creatinina_24h", "pearson_r"]
    if linha_creatinina.empty or float(linha_creatinina.iloc[0]) <= 0:
        raise RuntimeError("A coorte sintetica deveria associar creatinina maior a LOS maior.")

print(f"Arquivos em {saida.resolve()}")
print(f"X treino {x_treino.shape} | X teste {x_teste.shape}")
display(x_treino_esc.head())
